# llm weight

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, cohen_kappa_score

In [ ]:
llmdf = pd.read_excel("sample/llms_scored.xlsx")[
    ["instance_id", "cross_score", "gpt_5_mini_score"]
]
expertdf = pd.read_excel("expert_label.xlsx")[
    ["instance_id", "final_label", "final_score"]
]

origindf = expertdf.merge(llmdf, on="instance_id", validate="one_to_one")
assert len(origindf) == 320

In [ ]:
def get_table(df):
    rows = []
    for k in range(11):
        alpha = k / 10
        fused = (
            alpha * df["cross_score"]
            + (1 - alpha) * df["gpt_5_mini_score"]
        )

        predicted_label = np.digitize(
            fused, bins=[0.25, 0.50, 0.75]
        ) + 1

        rows.append({
            "alpha": alpha,
            "Spearman": spearmanr(fused, df["final_score"]).statistic,
            "MAE": mean_absolute_error(df["final_score"], fused),
            "QWK": cohen_kappa_score(
                df["final_label"], predicted_label, weights="quadratic"
            ),
        })

    return pd.DataFrame(rows)
    

table = get_table(origindf)

In [ ]:
print(table.to_latex(
    index=False,
    formatters={
        "alpha": lambda x: f"{x:.1f}",
        "Spearman": lambda x: f"{x:.3f}",
        "MAE": lambda x: f"{x:.3f}",
        "QWK": lambda x: f"{x:.3f}",
    }
))

In [ ]:
def line(results):
    

    fig, ax = plt.subplots(figsize=(5, 3))

    ax.plot(
        results["alpha"],
        results["Spearman"],
        marker="o",
        linewidth=1.5,
        markersize=4,
        label=r"Spearman ($\uparrow$)",
    )

    ax.plot(
        results["alpha"],
        results["MAE"],
        marker="s",
        linewidth=1.5,
        markersize=4,
        label=r"MAE ($\downarrow$)",
    )

    ax.plot(
        results["alpha"],
        results["QWK"],
        marker="^",
        linewidth=1.5,
        markersize=4,
        label=r"QWK ($\uparrow$)",
    )

    ax.axvline(
        0.2,
        color="red",
        linestyle="--",
        linewidth=1,
        label=r"$\alpha=0.2$",
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks(results["alpha"])
    ax.set_xlabel(r"Cross-encoder weight $\alpha$")
    ax.set_ylabel("Metric value")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, fontsize=8)

    fig.tight_layout()
line(table)